In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt 
from sklearn.feature_extraction.text import TfidfVectorizer
%matplotlib inline


In [2]:
df = pd.read_csv('messages.csv')

In [3]:
df.shape[0]

2893

In [4]:
df.shape[0]

2893

In [5]:
df['label'].value_counts("")
# 2412 : legitimate , 481 : spam 


label
0    2412
1     481
Name: count, dtype: int64

In [6]:
print(df.isna().value_counts()) # There are lot of places where these is only body and no subject
print(df.isna().sum())


subject  message  label
False    False    False    2831
True     False    False      62
Name: count, dtype: int64
subject    62
message     0
label       0
dtype: int64


In [7]:
df.fillna(' ', inplace=True) # dropping entries where subject is not available 
print(df.isna().sum())
 
from nltk.stem import SnowballStemmer
from nltk.stem import WordNetLemmatizer

snowball = SnowballStemmer("english")
lemmetiz = WordNetLemmatizer()

def preprocessing(para):
    # take array of string and applies stem and lemmeization to it 
    words = para.split()

    final = []
    for word in words: 
        
        # word = snowball.stem(word)
        word = lemmetiz.lemmatize(word, 'v')
        final.append(word)

    final_string = " ".join(final)
    return final_string


subject    0
message    0
label      0
dtype: int64


In [8]:
sub = df['subject']
sub.apply(preprocessing)
body = df['message']
body.apply(preprocessing)


C:\Users\Nishant Shekhar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\nltk\data.py:1192: RuntimeWarning: Security Violation [pathsec.ZipFile]: Unauthorized path C:\Users\Nishant Shekhar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\Roaming\nltk_data\corpora\wordnet.zip
  ZipFile.__init__(self, filename)
C:\Users\Nishant Shekhar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\nltk\corpus\reader\wordnet.py:1201: RuntimeWarning: Security Violation [CorpusReader]: Unauthorized path C:\Users\Nishant Shekhar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\Roaming\nltk_data\corpora\wordnet.zip\wordnet\lexnames
  with self.open("lexnames") as fp:
C:\Users\Nishant Shekhar\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packag

0       content - length : 3386 apple-iss research cen...
1       lang classification grime , joseph e . and bar...
2       i be post this inquiry for sergei atamas ( sat...
3       a colleague and i be research the differ degre...
4       earlier this morning i be on the phone with a ...
                              ...                        
2888    hello thank for stop by ! ! we have take many ...
2889    the list owner of : " kiddin " have invite you...
2890    judge from the return post , i must have sound...
2891    gotcha ! there be two separate fallacies in th...
2892    hello ! i ' m work on a thesis concern attitud...
Name: message, Length: 2893, dtype: object

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

subject_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english'
)

body_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english'
)

tfidf_sub = subject_vectorizer.fit_transform(sub)

tfidf_body = body_vectorizer.fit_transform(body)

print(tfidf_sub.toarray())
print(tfidf_body.toarray())

[[0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]
 [0. 0. 0. ... 0. 0. 0.]]
[[0.         0.         0.         ... 0.         0.         0.        ]
 [0.09420699 0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]


In [10]:
from scipy.sparse import hstack
input = hstack([tfidf_sub , tfidf_body])

In [11]:
print(type(input))
print(input.shape)

input = input.toarray()
X = pd.DataFrame(input)
X.to_csv('X.csv')


<class 'scipy.sparse._csr.csr_matrix'>
(2893, 63637)


In [12]:
y = df['label'].values
y = pd.DataFrame(y)
y.to_csv("y.csv", index=False)

In [13]:
import pickle 
with open("tfidf_subject.pkl", "wb") as f:
    pickle.dump(subject_vectorizer, f)

with open("tfidf_body.pkl", "wb") as f:
    pickle.dump(body_vectorizer, f)